# Khmer Tokenizer Comparison

Paste one Khmer legal article into the text block, then run the comparison.

In [ ]:
%pip install "khmercut[nn]" khmer-nltk datasets pandas pyarrow

In [ ]:
from khmercut import tokenize as cut_crf
from khmercut.nn import tokenize as cut_nn
from khmernltk import word_tokenize as nltk_cut
import time


def flatten_tokens(value):
    if isinstance(value, str):
        return [value]

    tokens = []
    for item in value:
        tokens.extend(flatten_tokens(item))
    return tokens


def clean_tokens(value):
    return [token for token in flatten_tokens(value) if token and not token.isspace()]


tokenizers = {
    "khmercut CRF": lambda text: cut_crf(text),
    "khmercut Neural": lambda text: cut_nn(text, deep=True),
    "khmer-nltk": lambda text: nltk_cut(text, return_tokens=True),
}

## Paste one article here

In [ ]:
article_text = """
គណបក្សនយោបាយដែលបានចុះបញ្ជីស្របតាមច្បាប់ស្តីពីគណបក្សនយោបាយ ហើយមានបំណងចូលរួមប្រកួតប្រជែងនៅក្នុងការបោះឆ្នោតជ្រើសរើសក្រុមប្រឹក្សា ត្រូវចាប់ផ្តើមចុះបញ្ជីគណបក្សនយោបាយឈរឈ្មោះបោះឆ្នោត តាមនីតិវិធីដែលរៀបចំដោយគណៈកម្មាធិការជាតិរៀបចំការបោះឆ្នោត យ៉ាងតិចប៉ែតសិប(៨០)ថ្ងៃមុនកាលបរិច្ឆេទនៃការបោះឆ្នោត ។
""".strip()

if not article_text or article_text.startswith("បិទភ្ជាប់"):
    raise ValueError("Paste one real Khmer legal article into article_text first.")

print(f"Characters: {len(article_text):,}")
print(article_text)

In [ ]:
for name, tokenizer in tokenizers.items():
    try:
        # Warm-up
        tokenizer(article_text)

        start = time.perf_counter()
        tokens = clean_tokens(tokenizer(article_text))
        elapsed = (time.perf_counter() - start) * 1000

        print(f"\n{name}")
        print(f"Token count: {len(tokens)}")
        print(f"Time: {elapsed:.2f} ms")
        print(tokens)

    except Exception as e:
        print(f"\n{name}: {type(e).__name__}: {e}")

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
TESTABLE_DIR = PROJECT_ROOT / 'data-pipeline' / 'testable'

from datasets import load_dataset

ds = load_dataset("khopilot/khmer-lexicon")

# Save locally
ds.save_to_disk(DATA_DIR / 'khmer-lexicon')